In [ ]:
#Calculation of total number of unwrapped base pair at each window.

In [ ]:
import MDAnalysis as mda
import numpy as np
from MDAnalysis.analysis import align

# -----------------------------------------------------------
# Compute Base-Pair COMs (Chain A paired with Chain B reverse)
# -----------------------------------------------------------
def basepair_com_3d(u, n_bp):
    coms = []
    for i in range(n_bp):
        resid_A = i + 1           # A:1 → A:147
        resid_B = n_bp - i        # B:147 → B:1

        selA = u.select_atoms(f"chainID A and resid {resid_A}")
        selB = u.select_atoms(f"chainID B and resid {resid_B}")

        if len(selA) == 0 or len(selB) == 0:
            raise ValueError(f"Missing atoms for A:{resid_A} or B:{resid_B}")

        combined = selA + selB
        com = combined.center_of_mass()
        coms.append(com)

    return np.array(coms)


# -----------------------------------------------------------
# User parameters
# -----------------------------------------------------------
topology = "nucl.pdb"
n_windows = 13   # windows 0 → 12

n_bp = 147
threshold = 7.0          # Å
end_bp_count = 30        # only ends
n_blocks = 5             # block averaging


# Basepair indices to check (first 30 + last 30)
first_end = set(range(0, end_bp_count))
last_end  = set(range(n_bp - end_bp_count, n_bp))


# -----------------------------------------------------------
# Output storage
# -----------------------------------------------------------
window_numbers = []
avg_bp_values = []
errors = []


# -----------------------------------------------------------
# Loop over umbrella windows
# -----------------------------------------------------------
for i in range(n_windows):

    xtc_file = f"umb{i}_upto50ns.nucl.pbc.xtc"
    print(f"\n-------------------------------")
    print(f"Processing window {i}: {xtc_file}")
    print(f"-------------------------------")

    u_ref = mda.Universe(topology)
    u_traj = mda.Universe(topology, xtc_file)

    # ---------- align trajectory ----------
    aligner = align.AlignTraj(u_traj, u_ref, select="name P", in_memory=True)
    aligner.run()

    # ---------- reference COMs ----------
    ref_coms = basepair_com_3d(u_ref, n_bp)

    # ---------- compute unwrapped per frame ----------
    unwrapped_counts = []
    for ts in u_traj.trajectory:

        traj_coms = basepair_com_3d(u_traj, n_bp)
        distances = np.linalg.norm(traj_coms - ref_coms, axis=1)

        count = 0
        for bp_idx, dist in enumerate(distances):
            if dist > threshold and (bp_idx in first_end or bp_idx in last_end):
                count += 1

        unwrapped_counts.append(count)

    unwrapped_counts = np.array(unwrapped_counts)
    n_frames = len(unwrapped_counts)

    # ---------- block averaging ----------
    block_size = n_frames // n_blocks
    block_means = []

    for b in range(n_blocks):
        start = b * block_size
        end = (b + 1) * block_size if b < n_blocks - 1 else n_frames
        block_means.append(np.mean(unwrapped_counts[start:end]))

    block_means = np.array(block_means)

    mean_val = np.mean(block_means)
    std_val = np.std(block_means, ddof=1)
    error_val = std_val / np.sqrt(n_blocks)

    # Store final results
    window_numbers.append(i)
    avg_bp_values.append(mean_val)
    errors.append(error_val)

    print(f"Window {i}: Mean Nbp = {mean_val:.3f}, Error = {error_val:.3f}")


# -----------------------------------------------------------
# Save summary for all windows
# -----------------------------------------------------------
with open("unwrapped_bp_summary_windows.txt", "w") as f:
    f.write("Window\tAvg_Nbp\tError\n")
    for w, mean_v, err_v in zip(window_numbers, avg_bp_values, errors):
        f.write(f"{w}\t{mean_v:.6f}\t{err_v:.6f}\n")

print("\nAll window results written to: unwrapped_bp_summary_windows.txt")
